In [24]:
import numpy as np
from os import listdir
import os
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from os import listdir
import matplotlib.patches as mpatches
from scipy import signal
import scipy.stats as scistats
from scipy.stats import chi2
import gsw
import geopy.distance
import signalz
import scipy
from scipy import signal
from scipy.interpolate import interp1d
from scipy.interpolate import BarycentricInterpolator
import math
#import jmkxarray
from matplotlib import ticker
from matplotlib.lines import Line2D
import scipy.stats as stats

In [3]:
all_sections_combined = xr.open_dataset('/Users/Lauryn/THESIS/CPROOF_Work/Files/good_files/all_files15.nc')

In [4]:
climate = '/Users/Lauryn/THESIS/CPROOF_Work/Climatology/LineP_climatology_1956to2012.nc'  

with xr.open_dataset(climate) as climate:

    #find distance from Station 1
    climate['distance'] =[geopy.distance.geodesic((48.56127328, -125.53143223), (climate.latitude[i],climate.longitude[i])).km for i in range(len(climate.longitude))]


In [5]:
transfer_path = '/Users/Lauryn/THESIS/CPROOF_Work/Files/transfer_functions/'

transfer = [ 'walle2019_jul_WE_transfer_functions.nc', 'walle2019_dec_EW_transfer_functions.nc',
            'walle2019_dec_WE_transfer_functions.nc','walle2021_EW_transfer_functions.nc',
            'walle2021_WE_transfer_functions.nc','rosie2022_EW_transfer_functions.nc',
            'rosie2022_WE_transfer_functions.nc','rosie2023_EW_transfer_functions.nc', 
            'rosie2023_WE_transfer_functions.nc','walle2023_EW_transfer_functions.nc', 
            'walle2023_WE_transfer_functions.nc','rosie2024_EW_transfer_functions.nc',
            'rosie2024_WE_transfer_functions.nc','walle2024_EW_transfer_functions.nc', 
            'walle2024_EW_transfer_functions.nc']


# Spectra/transfer code

Code from jmkxarray (https://github.com/jklymak/jmkxarray) 

In [7]:
import logging
_log = logging.getLogger(__name__)

def gappy_interp(ds, *, dim='', xgrid=None, maxgap=np.inf):
    """

    OBSOLETE: use `ds.interpolate_na(maxgap=)
    interp ds on xgrid along dimension dim.  However don't
    interpolate across gaps larger than maxgap.



    Parameters
    ----------
    ds : xarray.DataSet or xarray.DataFrame
        data frame.  Must have *dim* as a dimension
    dim : string
        dimension to interpolate on.
    xgrid : array
        grid to interpolate to in dimension *dim*
    maxgap : float
        maximum size of gap in *dim* to interpolate across

    Returns
    -------
    DataSet or DataArray with dim=xgrid.

    Examples
    --------
    Note in the below that the gap from x=1 to 2.1 is _not_ interpolated over.

    >>> da = xr.DataArray(
        ...     data=[[1, 2, 3, 4, 5, 6]],
        ...     dims=("x"),
        ...     coords={"x": [0, 0.5, 1, 2.1, 2.5, 3.4]},
        ... )
    >>> dn = gappy_interp(da, dim='x', xgrid=np.arange(0, 3.1, 0.2), maxgap=1.0)
    >>> dn
    <xarray.DataArray (x: 16)>
    array([1.        , 1.4       , 1.8       , 2.2       , 2.6       ,
           3.        ,        nan,        nan,        nan,        nan,
           nan, 4.25      , 4.75      , 5.11111111, 5.33333333, 5.55555556])
    Coordinates:
    * x        (x) float64 0.0 0.2 0.4 0.6 0.8 1.0 1.2 ... 2.0 2.2 2.4 2.6 2.8 3.0
    """

    x0 = ds[dim].values
    print('xgrid:', xgrid)
    print('dim', dim)
    ds = ds.interp(**{dim:xgrid})
    dx = np.diff(x0)
    bad = (dx > maxgap).nonzero()[0]
    print("Bad", bad)
    mask = np.isfinite(xgrid)
    for b in bad:
        print("B", b, x0[b], x0[b+1])
        mask[(xgrid > x0[b]) & (xgrid < x0[b+1])] = False
    mask = xr.DataArray(mask, dims=(dim))
    return ds.where(mask)


def depth_to_iso(ds, pden='pden', depths='depths', xdim='along',
                 pden0=None, isodepths=None):
    """
    Map fields in this dataset to potential density coordinates.

    Parameters
    -----------

    ds : xarray Dataset
        Dataset that at least has *pden* and *depths* fields to map to

    pden : str (default 'pden')
        Name of Array in *ds* that has the density information.

    depths : str (default 'depths')
        Name of depth coordinate that has depth information.

    xdim : str (default 'along')
        Name of x coordinate of the data set

    pden0 : array-like (default None)
        Array of mean densities to interpolate onto.  If not provided, the
        data set mean is used.

    isodepths : array-like (default None)
        Mean depths of the isopycnals defined in *pden0*.
    """

    if pden0 is None:
        pden0 = ds[pden].mean(dim=xdim)
        isodepths = ds[depths].where(pden0 > 0).dropna(dim=depths)
        pden0 = pden0.dropna(dim=depths)

    M = pden0.shape[0]
    N = ds[xdim].shape[0]
    print(M, N)

    dsiso = xr.Dataset(coords={'isodepths':('isodepths', isodepths.values),
                               xdim: (xdim, ds[xdim].values)})
    print(dsiso)
    print(len(isodepths.values))
    for td in ds.variables:
        _log.info('todo', td, ds[td].shape)
        print(td, ds[td].shape)
        if td == xdim:
            pass
        elif td == depths:
            pass
        elif ds[td].shape == ds[xdim].shape:
            dsiso[td] = (xdim, ds[td].values)
        elif ds[td].shape == ds[pden].shape:
            dsiso[td] = (('isodepths', xdim), np.zeros((M, N)))
            for i in range(N):
                dsiso[td][:, i] = np.interp(pden0, ds[pden][:, i], ds[td].values[:, i])
        else:
            try:
                dsiso[td] = ds[td]
            except ValueError:
                pass

    return dsiso


def get_n_blocks(nfft, N, noverlap_min=None):
    if noverlap_min is None:
        noverlap_min = nfft / 2
    for m in range(2, 2000):
        noverlap = np.ceil(- (N - m * nfft) / (m - 1))
        if noverlap > 0 and noverlap >= noverlap_min:
            total = m * nfft - (m-1) * noverlap
            if total == nfft:
                noverlap = 0
            return int(noverlap), int(total), m


def power_spec(da, nfft=256, xdim='along', ydim='depths', xunits='km',
               dataunits='kg/m^3'):

    """
    Calculate power spectra segments of a DataArray and return as a Dataset.

    Parameters
    ----------

    da : DataArray
        data array to calculate spectra over.  Spectra are calculated in the
        second dimension, usually something like "x" or "time".

    nfft : integer
        length of FFT blocks

    xdim : string
        dimension in DataArray to calculate spectra along.

    ydim : string
        dimension in DataArray over which we iterate to calculate the spectra.

    xunits : string
        units of the xdimension

    dataunits : string
        units of the data that is being analyzed

    Returns
    -------

    ps : DataSet
        Data set with coordinates (blocks, depths, kx), if ydim is "depths".  Note that you
        may want to rename the coordinates if kx is a frequency.  Note that kx has units of
        cycles / km, *not* rad / km.  The spectrum has units of V^2 / cpkm, if "V" are the
        units of the data in *da*, and is normalized so that the integral of S(kx) dkx is
        equal to the variance of the signal in *da*.

    """
    N = da[xdim].shape[0]
    window = np.hanning(nfft)
    wnorm = (window * window).sum()
    noverlap, total, nblocks = get_n_blocks(nfft, N, noverlap_min=nfft/2)
    _log.info(noverlap, nblocks)
    dx = da[xdim].diff(dim=xdim).median().values
    kx = np.arange(0, 1/2-1e-5, 1/nfft) / dx
    ps =  xr.Dataset(coords={'blocks': np.arange(nblocks), 'depths': da[ydim].values, 'kx': kx })
    ps['kx'].attrs = {'description': 'wavenumber or frequency',
                      'units': f'cycles per {xunits}'}
    ps['spectrum'] = (('blocks', 'depths', 'kx'), np.zeros((nblocks, da[ydim].shape[0], len(kx))))
    ps['spectrum'].attrs = {'description': 'un-averaged power spectra from overlaping blocks',
                            'units': f'({dataunits})^2 / cp{xunits}',
                            'source': f'{da.name}'}
    ps['along_start'] = (('blocks'), np.zeros(nblocks))
    ps['along_stop'] = (('blocks'), np.zeros(nblocks))
    ps['along_start'].attrs = {'description' : f'start of the block in {xdim}'}
    ps['along_stop'].attrs = {'description' : f'start of the block in {xdim}'}
    ps['blocks'].attrs = {'description' : 'block index for the spectral estimates'}

    start = 0
    for nn in range(nblocks):
        stop = start + nfft
        _log.info(start, stop, da.isel({xdim: slice(start, stop)}).shape)
        xx = da.isel({xdim: slice(start, stop)})
        xx = xx - xx.mean(dim=xdim)
        ff = np.fft.fft(xx * window, axis=1)
        pp = ff[:, :int(nfft/2)]*np.conj(ff[:, :int(nfft/2)]) * 2 * dx / wnorm
        ps['spectrum'][nn, :, : ] = np.real(pp)
        ps['along_start'][nn] = da[xdim].isel({xdim:start})
        ps['along_stop'][nn] = da[xdim].isel({xdim: stop-1})
        start = stop - noverlap
    ps.attrs = {'nblocks': nblocks, 'noverlap': noverlap, 'nfft': nfft}
    return ps


def multi_psd(da, minnfft=64, xdim='along', ydim='depths', xunits='km',
              dataunits='kg/m^3'):

    """
    Calculate power spectra segments of a DataArray with different resolutions,
    and return a spectra.  Interpolate over gaps that are a bit larger than the resolution
    changes

    nfft: size of fft blocks to try.
        Maxgap for the current block will depend on the size of the next block.
    """
    N = len(da[xdim])

    nffts = [64,64*2,64*2*2,64*2*2*2,64*2*2*2*2] 

    maxgaps = nffts[1:]
    maxgaps += [2]
    maxgaps = maxgaps[::-1]
    # get the maxgap to interpolate over for each fft attempt
    spec = []
    kx = []
    for nn, nfft in enumerate(nffts[::-1]):
        dd = da.interpolate_na(dim=xdim, max_gap=maxgaps[nn])

        p = power_spec(dd, nfft=int(nfft), xdim=xdim, ydim=ydim,
                        xunits=xunits, dataunits=dataunits).mean(dim='blocks')
        if nn == 0:
            kmax = None
            spec = p.spectrum.values[:, 1:]
            kx = p.kx.values[1:]
        else:
            kmax = kx[0] - dkx / 2
            spec = np.concatenate((p.spectrum.sel(kx=slice(None, kmax)).values[:, 1:], spec), axis=1)
            kx = np.concatenate((p.kx.sel(kx=slice(None, kmax)).values[1:], kx))
        dkx = p.kx.diff(dim='kx').median(dim='kx')
    dout = xr.Dataset(coords={'depths': da[ydim].values, 'kx': kx })
    dout['kx'] = ('kx', kx)
    dout['spectrum'] = (('depths', 'kx'), spec)

    return dout


def whiten(kx, sp):
    return sp*(np.pi*2*kx)**2

In [8]:
#### break data into nearshore and offshore before doing spectra stuff
def block_avg_normalized(file, file_transfer):
    ##open_data
    section = file.isel(distance=slice(None, None, -1))
    section_transfer = xr.open_dataset(transfer_path + file_transfer)

    ### block average PSD
    section_block = multi_psd(((section.iso_temps-all_sections_combined.pot_temp_mean)/all_sections_combined.pot_temp_std).T, minnfft=64, 
                            xdim='distance', ydim='depth', xunits='km', dataunits='C')

    ### interpolate transfer function onto blocked k grid 
    block_transfer = np.empty((1100,511))
    block_transfer[:]=np.nan

    for i in range(0,1100):
        block_transfer[i,:] = np.interp(section_block.kx,section_transfer.k,
                              section_transfer.transfer_fn[i].values) 


    ### divide PSD by transfer functions 
    block_PSD = np.empty((1000,511))
    block_PSD[:]= np.nan

    for i in range(0,1000):
        block_PSD[i,:] =  section_block.spectrum[i]/block_transfer[i]

    block_PSD = xr.DataArray(block_PSD, dims=('depth','k'))

    block_PSD = block_PSD.rename('blocked_iso_potential_temp_PSD')
    block_PSD['depth'] = section.depth[:1000]
    block_PSD['k'] = section_block.kx.values
    
    return block_PSD, block_PSD.k, block_PSD.depth



Nearshore/offshore

In [10]:
spectra_all = np.empty((1000,511,15))
spectra_all[:]=np.nan


for i in range(15):
    file = all_sections_combined.isel(file=i,depth=slice(0,1000)).sel(distance=slice(-80,-550))   
    spec,kk,dpth = block_avg_normalized(file,transfer[i])
    spectra_all[:,:,i]=spec
                                
near_spectra = xr.Dataset(
    {
        "spectra": (("depth", "k","file"), spectra_all[:,1::,:]),  
    },
    coords={
        "k": kk.values[1::],
        "depth": dpth.values,  
        "file": np.arange(0,15,1)
        
    }
)
spectra_all = np.empty((1000,511,15))
spectra_all[:]=np.nan


for i in range(15):
    file = all_sections_combined.isel(file=i,depth=slice(0,1000)).sel(distance=slice(-550,-1178))   
    spec,kk,dpth = block_avg_normalized(file,transfer[i])
    spectra_all[:,:,i]=spec
                                
off_spectra = xr.Dataset(
    {
        "spectra": (("depth", "k","file"), spectra_all[:,1::,:]),  
    },
    coords={
        "k": kk.values[1::],
        "depth": dpth.values,  
        "file": np.arange(0,15,1)
        
    }
)


In [ ]:
model_path = '/Users/Lauryn/THESIS/CPROOF_Work/NEPOM/'


all_synthetic = xr.open_dataset(model_path+'all_synthetic_files.nc')

#### make nearshore and offshore spectra 
def block_avg_normalized(section, glider):
    section = section.isel(distance=slice(None,None,-1))
    glider = glider.isel(distance=slice(None,None,-1))


    anom = ((section.T- glider.pot_temp_mean.values)/
                  glider.pot_temp_std.values)
    section_block = multi_psd(anom.T ,minnfft=64, 
                            xdim='distance', ydim='depth', xunits='km', dataunits='C')
    
    section_block = section_block.rename({'depths': 'depth','kx': 'k'})
    section_block['blocked_iso_potential_temp_PSD'] = section_block.spectrum

    return section_block 

In [ ]:
spectra_all = np.empty((1000,511,15))
spectra_all[:]=np.nan

glider = all_sections_combined.isel(file=1,depth=slice(0,1000)).sel(distance=slice(-80,-550))   


for i in range(15):
    spec = block_avg_normalized(all_synthetic.isel(file=i).sel(distance=slice(-80,-550)).iso_temps,glider)
    spectra_all[:,:,i]=spec.blocked_iso_potential_temp_PSD
                                
model_near_spectra = xr.Dataset(
    {
        "spectra": (("depth", "k","file"), spectra_all[:,1::,:]),  
    },
    coords={
        "k": spec.k[1::],
        "depth": spec.depth,  
        "file": np.arange(0,15,1)
        
    }
)

spectra_all = np.empty((1000,511,15))
spectra_all[:]=np.nan

glider = all_sections_combined.isel(file=1,depth=slice(0,1000)).sel(distance=slice(-550,-1200))   


for i in range(15):
    spec = block_avg_normalized(all_synthetic.isel(file=i).sel(distance=slice(-550,-1200)).iso_temps,glider)
    spectra_all[:,:,i]=spec.blocked_iso_potential_temp_PSD
                                
model_off_spectra = xr.Dataset(
    {
        "spectra": (("depth", "k","file"), spectra_all[:,1::,:]),  
    },
    coords={
        "k": spec.k[1::],
        "depth": spec.depth,  
        "file": np.arange(0,15,1)
        
    }
)

Full transect 

In [21]:
def block_avg_normalized_glider(file, file_transfer, size):

    # Reverse order
    section = file.isel(distance=slice(None, None, -1))
    section_transfer = xr.open_dataset(transfer_path + file_transfer)

    # Normalize to anomalies
    anom = ((section - all_sections_combined.pot_temp_mean) /
            all_sections_combined.pot_temp_std).T

    nfft = 3052
    section_block = power_spec(anom, nfft=nfft,
                               xdim='distance', ydim='depth',
                               xunits='km', dataunits='C').mean(dim='blocks')

    section_block = section_block.rename({'depths': 'depth', 'kx': 'k'})
    kx = section_block.k.values
    n_kx = kx.size

    block_transfer = np.full((1100, size), np.nan)
    block_PSD = np.full((1000, size), np.nan)

    # === Interpolate transfer function onto current kx and normalize PSD ===
    for i in range(1100):
        transfer_vals = np.interp(kx, section_transfer.k, section_transfer.transfer_fn[i].values)
        block_transfer[i, :n_kx] = transfer_vals

    for i in range(1000):
        psd_vals = section_block.spectrum[i].values
        block_PSD[i, :n_kx] = psd_vals / block_transfer[i, :n_kx]

    block_PSD = xr.DataArray(block_PSD, dims=('depth', 'k'))
    block_PSD.name = 'blocked_iso_potential_temp_PSD'
    block_PSD['depth'] = section.depth[:1000]
    block_PSD['k'] = np.linspace(kx[0], kx[-1], size)  # consistent dummy axis

    return block_PSD, block_PSD.k, block_PSD.depth

def chi2_error_single_segment(psd: xr.DataArray, nu=2, ci=0.68):
    """
    Compute confidence intervals for a single-segment PSD assuming chi-squared distribution.

    Parameters
    ----------
    psd : xr.DataArray
        Power spectrum
    nu : int
        Degrees of freedom (2 for single segment)
    ci : float
        Confidence interval (e.g., 0.68 for 68%)

    Returns
    -------
    lower, upper : xr.DataArray
        Lower and upper confidence interval bounds
    """
    alpha = 1 - ci
    lower = psd * stats.chi2.ppf(alpha / 2, df=nu) / nu
    upper = psd * stats.chi2.ppf(1 - alpha / 2, df=nu) / nu
    return lower, upper


In [25]:
n_files = 15
depth_dim = 1000
k_dim = 2175

# Preallocate arrays
lower_all = np.full((depth_dim, k_dim, n_files), np.nan)
upper_all = np.full((depth_dim, k_dim, n_files), np.nan)

for i in range(n_files):
    print(f"Processing file {i}")
    try:
        psd, kx, dpth = block_avg_normalized_glider(
            all_sections_combined.isel(file=i, depth=slice(0, 1000)).sel(distance=slice(-80, -1178)),
             transfer[i], 2176
        )
        # Compute CI bounds using chi-squared method
        lower, upper = chi2_error_single_segment(psd[:,1:])

        lower_all[:, :, i] = lower
        upper_all[:, :, i] = upper

        if i == 0:
            k_used = kx.values
            depth_used = dpth.values

    except Exception as e:
        print(f"File {i} failed: {e}")

spectra_ci = xr.Dataset(
    {
        "ci_lower": (("depth", "k", "file"), lower_all),
        "ci_upper": (("depth", "k", "file"), upper_all),
    },
    coords={
        "k": k_used[1::],
        "depth": depth_used,
        "file": np.arange(n_files)
    }
)

spectra_ci.to_netcdf("glider_spectra_confidence_bounds.nc")

Processing file 0
Processing file 1
Processing file 2
Processing file 3
Processing file 4
Processing file 5
Processing file 6
Processing file 7
Processing file 8
Processing file 9
Processing file 10
Processing file 11
Processing file 12
Processing file 13
Processing file 14
